In [ ]:
import pandas as pd 

In [ ]:
# reading the data 
df = pd.read_csv("input-data/Project 2 - BRPMM/process A/calibration data/calibration_data.csv")
df.head()

In [ ]:
# checking for missing values with count
df.describe().T

In [ ]:
# reading the labels 
df_labels = pd.read_csv("input-data/Project 2 - BRPMM/process A/calibration data/calibration_labels.csv")
df_labels.head()

In [ ]:
# checking for missing values in the labels
df_labels.describe().T

In [ ]:
# seeing whether the data is time series 
import matplotlib.pyplot as plt
import seaborn as sns

df["batch_id"].value_counts().plot.bar(); plt.show()
df.groupby("batch_id").size().plot.hist(bins=30); plt.title("rows per batch"); plt.show()

In [ ]:
# merging the data and the labels
df = df.merge(df_labels, on="batch_id", how="left")
df["label"].isna().sum()   # rows that didn't get a label: should be 0, or explainable

In [ ]:
# inspecting the new merged data
df.head()

In [ ]:
df["t"] = df.groupby("batch_id").cumcount()   # 0, 1, 2, ... restarts for each batch

In [ ]:
df.head()

In [ ]:
# counting the time series length of each batch
lengths = df.groupby("batch_id").size()
lengths.describe()
lengths.value_counts().head()

In [ ]:
# creating a dataframe with one row per batch, making features, creating "borderline" batches
import numpy as np

features = ["flow_rate_1", "density", "flow_rate_2", "pressure_1", "temperature_1",
            "temperature_2", "temperature_3", "temperature_4",
            "pressure_2", "pressure_3", "pressure_4"]

batch_feats = df.groupby("batch_id")[features].agg(["mean", "std", "min", "max"])
batch_feats.columns = ["_".join(c) for c in batch_feats.columns]

t_c = df["t"] - df.groupby("batch_id")["t"].transform("mean")
x_c = df[features] - df.groupby("batch_id")[features].transform("mean")
slopes = x_c.mul(t_c, axis=0).groupby(df["batch_id"]).sum().div(
    (t_c ** 2).groupby(df["batch_id"]).sum(), axis=0)
slopes.columns = [f"{f}_slope" for f in features]

batch_feats = batch_feats.join(slopes).join(df_labels.set_index("batch_id"))

# mark the borderline batches from the brief
batch_feats["group"] = batch_feats["label"]
batch_feats.loc[batch_feats.index.isin(["A_101", "A_102", "A_103", "A_104"]), "group"] = "borderline"

print(batch_feats.shape)
batch_feats["group"].value_counts()

In [ ]:
# inspecting the time series, plotting. good batches are in blue, bad are in red
import matplotlib.pyplot as plt

fig, axes = plt.subplots(4, 3, figsize=(15, 12))
for ax, f in zip(axes.flat, features):
    for b, g in df.groupby("batch_id"):
        rej = g["label"].iloc[0] == "rejected"
        ax.plot(g["t"][::10], g[f][::10], color="red" if rej else "blue",
                alpha=0.5 if rej else 0.15, lw=0.7)
    ax.set_title(f)
plt.tight_layout(); plt.show()

In [ ]:
# creating a PCA map of all batches 
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import seaborn as sns

feat_cols = [c for c in batch_feats.columns if c not in ["label", "group"]]
X = batch_feats[feat_cols]
acc = batch_feats["label"] == "accepted"

scaler = StandardScaler().fit(X[acc])      # fit on accepted only
pca = PCA(n_components=2).fit(scaler.transform(X[acc]))
Z = pca.transform(scaler.transform(X))

sns.scatterplot(x=Z[:, 0], y=Z[:, 1], hue=batch_feats["group"], s=50)
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%})")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%})")
plt.title("Each dot = one batch"); plt.show()

In [ ]:
# leave-one-batch-out anomaly detection
# The model only ever learns from accepted batches. 
# Each accepted batch gets scored by a model that never saw it. 
# Rejected batches are scored by every fold's model and averaged.

from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score

X_acc, X_rej = X[acc], X[~acc]
acc_scores = pd.Series(index=X_acc.index, dtype=float)
rej_scores = []

for b in X_acc.index:                           # leave one accepted batch out
    train = X_acc.drop(index=b)
    sc = StandardScaler().fit(train)
    model = IsolationForest(n_estimators=300, random_state=0).fit(sc.transform(train))
    acc_scores[b] = -model.score_samples(sc.transform(X_acc.loc[[b]]))[0]
    rej_scores.append(-model.score_samples(sc.transform(X_rej)))

rej_scores = pd.Series(np.mean(rej_scores, axis=0), index=X_rej.index)

scores = pd.concat([acc_scores, rej_scores])       # higher = more anomalous
y = (batch_feats.loc[scores.index, "label"] == "rejected").astype(int)
print("ROC AUC:", round(roc_auc_score(y, scores), 3))

In [ ]:
# picking a threshold and counting the mistakes
threshold = acc_scores.quantile(0.70)   # allow ~5% false alarms on good batches

pred = (scores > threshold).astype(int)
tp = ((pred == 1) & (y == 1)).sum(); fn = ((pred == 0) & (y == 1)).sum()
fp = ((pred == 1) & (y == 0)).sum(); tn = ((pred == 0) & (y == 0)).sum()
print(f"Caught {tp} of {tp+fn} rejected batches, missed {fn}")
print(f"False alarms: {fp} of {fp+tn} accepted batches")

plt.hist(acc_scores, bins=30, alpha=0.6, label="accepted")
plt.hist(rej_scores, bins=30, alpha=0.6, label="rejected", color="red")
plt.axvline(threshold, color="k", ls="--", label="threshold")
plt.xlabel("anomaly score"); plt.legend(); plt.show()

print(scores[scores.index.isin(["A_101","A_102","A_103","A_104"])])  # where do the borderline ones land?

In [ ]:
from sklearn.base import clone, BaseEstimator
from sklearn.metrics import roc_auc_score, average_precision_score

BORDER = ["A_101", "A_102", "A_103", "A_104"]

def evaluate(model, X, acc_mask):
    """Leave-one-accepted-batch-out. Returns anomaly scores (higher = more anomalous)."""
    X_acc, X_rej = X[acc_mask], X[~acc_mask]
    acc_s = pd.Series(index=X_acc.index, dtype=float)
    rej_s = []
    for b in X_acc.index:
        m = clone(model).fit(X_acc.drop(index=b))
        acc_s[b] = -m.score_samples(X_acc.loc[[b]])[0]
        rej_s.append(-m.score_samples(X_rej))
    rej_s = pd.Series(np.mean(rej_s, axis=0), index=X_rej.index)
    return acc_s, rej_s

def summarize(acc_s, rej_s, q=0.95):
    scores = pd.concat([acc_s, rej_s])
    y = scores.index.isin(rej_s.index).astype(int)
    thr = acc_s.quantile(q)
    border = rej_s[rej_s.index.isin(BORDER)]
    return {
        "ROC AUC": round(roc_auc_score(y, scores), 3),
        "PR AUC": round(average_precision_score(y, scores), 3),
        "caught": f"{(rej_s > thr).sum()}/{len(rej_s)}",
        "false alarms": f"{(acc_s > thr).sum()}/{len(acc_s)}",
        "borderline caught": f"{(border > thr).sum()}/{len(border)}",
    }

In [ ]:
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors

class KMeansDetector(BaseEstimator):
    """Anomaly score = distance to the nearest cluster centre."""
    def __init__(self, n_clusters=3):
        self.n_clusters = n_clusters
    def fit(self, X, y=None):
        self.km_ = KMeans(self.n_clusters, n_init=10, random_state=0).fit(X)
        return self
    def score_samples(self, X):
        return -self.km_.transform(X).min(axis=1)   # higher = more normal

class FastABOD(BaseEstimator):
    """Angle-based outlier detection on k nearest neighbours."""
    def __init__(self, n_neighbors=10):
        self.n_neighbors = n_neighbors
    def fit(self, X, y=None):
        self.X_ = np.asarray(X)
        self.nn_ = NearestNeighbors(n_neighbors=self.n_neighbors).fit(self.X_)
        return self
    def score_samples(self, X):
        X = np.asarray(X)
        _, idx = self.nn_.kneighbors(X)
        out = []
        for x, nb in zip(X, idx):
            D = self.X_[nb] - x
            n2 = (D ** 2).sum(axis=1) + 1e-12
            i, j = np.triu_indices(len(D), k=1)
            vals = (D[i] * D[j]).sum(axis=1) / (n2[i] * n2[j])
            out.append(vals.var())
        return np.array(out)                         # higher = more normal                     # higher = more normal

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import LocalOutlierFactor
from sklearn.covariance import EllipticEnvelope
from sklearn.svm import OneClassSVM
from sklearn.linear_model import SGDOneClassSVM
from sklearn.kernel_approximation import Nystroem
from sklearn.ensemble import IsolationForest
import warnings; warnings.filterwarnings("ignore")

models = {
    "Isolation Forest": make_pipeline(StandardScaler(),
                          IsolationForest(n_estimators=300, random_state=0)),
    "LOF":              make_pipeline(StandardScaler(),
                          LocalOutlierFactor(n_neighbors=10, novelty=True)),
    "Robust covariance": make_pipeline(StandardScaler(), PCA(n_components=10),
                          EllipticEnvelope(random_state=0)),
    "One-class SVM":    make_pipeline(StandardScaler(),
                          OneClassSVM(kernel="rbf", gamma="scale", nu=0.05)),
    "One-class SVM (SGD)": make_pipeline(StandardScaler(),
                          Nystroem(gamma=0.01, n_components=100, random_state=0),
                          SGDOneClassSVM(nu=0.05, random_state=0)),
    "K-means":          make_pipeline(StandardScaler(), KMeansDetector(n_clusters=3)),
    "ABOD":             make_pipeline(StandardScaler(), FastABOD(n_neighbors=10)),
}

all_scores, results = {}, {}
for name, model in models.items():
    print("running", name, "...")
    acc_s, rej_s = evaluate(model, X, acc)
    all_scores[name] = (acc_s, rej_s)
    results[name] = summarize(acc_s, rej_s)

results_df = pd.DataFrame(results).T.sort_values("ROC AUC", ascending=False)
results_df

In [ ]:
from sklearn.metrics import roc_curve

for name, (acc_s, rej_s) in all_scores.items():
    scores = pd.concat([acc_s, rej_s])
    y = scores.index.isin(rej_s.index).astype(int)
    fpr, tpr, _ = roc_curve(y, scores)
    plt.plot(fpr, tpr, label=f"{name} ({results[name]['ROC AUC']})")
plt.plot([0, 1], [0, 1], "k--", lw=0.8)
plt.xlabel("false alarm rate"); plt.ylabel("detection rate"); plt.legend(fontsize=8)
plt.title("Leave-one-batch-out ROC, process A"); plt.show()

In [ ]:
import torch, torch.nn as nn
from sklearn.model_selection import KFold

L = 100
def to_fixed_length(g):
    return np.stack([blk.mean(axis=0) for blk in np.array_split(g[features].to_numpy(), L)])

ids = batch_feats.index
S = np.stack([to_fixed_length(df[df["batch_id"] == b]) for b in ids])  # (161, 100, 11)
lab = batch_feats["label"].values

class LSTMAE(nn.Module):
    def __init__(self, n_feat, hidden=32):
        super().__init__()
        self.enc = nn.LSTM(n_feat, hidden, batch_first=True)
        self.dec = nn.LSTM(hidden, hidden, batch_first=True)
        self.out = nn.Linear(hidden, n_feat)
    def forward(self, x):
        _, (h, _) = self.enc(x)
        z = h[-1].unsqueeze(1).repeat(1, x.size(1), 1)
        y, _ = self.dec(z)
        return self.out(y)

def train_ae(Xtr, epochs=100):
    torch.manual_seed(0)
    m = LSTMAE(Xtr.shape[2]); opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    Xt = torch.tensor(Xtr, dtype=torch.float32)
    for _ in range(epochs):
        for i in torch.randperm(len(Xt)).split(16):
            loss = ((m(Xt[i]) - Xt[i]) ** 2).mean()
            opt.zero_grad(); loss.backward(); opt.step()
    return m

def recon_error(m, A):
    with torch.no_grad():
        At = torch.tensor(A, dtype=torch.float32)
        return ((m(At) - At) ** 2).mean(dim=(1, 2)).numpy()

acc_idx, rej_idx = np.where(lab == "accepted")[0], np.where(lab == "rejected")[0]
acc_s = pd.Series(index=ids[acc_idx], dtype=float); rej_list = []

for k, (tr, te) in enumerate(KFold(5, shuffle=True, random_state=0).split(acc_idx)):
    print("fold", k + 1)
    mu = S[acc_idx[tr]].mean(axis=(0, 1)); sd = S[acc_idx[tr]].std(axis=(0, 1)) + 1e-8
    m = train_ae((S[acc_idx[tr]] - mu) / sd)
    acc_s.iloc[te] = recon_error(m, (S[acc_idx[te]] - mu) / sd)
    rej_list.append(recon_error(m, (S[rej_idx] - mu) / sd))

rej_s = pd.Series(np.mean(rej_list, axis=0), index=ids[rej_idx])
all_scores["LSTM autoencoder"] = (acc_s, rej_s)
results["LSTM autoencoder"] = summarize(acc_s, rej_s)
pd.DataFrame(results).T.sort_values("ROC AUC", ascending=False)

In [ ]:
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,
                             balanced_accuracy_score, precision_score, recall_score,
                             f1_score, fbeta_score, matthews_corrcoef, confusion_matrix)

def summarize(acc_s, rej_s, q=0.95):
    scores = pd.concat([acc_s, rej_s])
    y = scores.index.isin(rej_s.index).astype(int)      # 1 = rejected (positive class)
    thr = acc_s.quantile(q)
    pred = (scores > thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    border = rej_s[rej_s.index.isin(BORDER)]

    return {
        # threshold-free
        "ROC AUC": roc_auc_score(y, scores),
        "PR AUC": average_precision_score(y, scores),
        # at the chosen threshold
        "accuracy": accuracy_score(y, pred),
        "balanced accuracy": balanced_accuracy_score(y, pred),
        "precision": precision_score(y, pred, zero_division=0),
        "recall (sensitivity)": recall_score(y, pred, zero_division=0),
        "specificity": tn / (tn + fp),
        "F1": f1_score(y, pred, zero_division=0),
        "F2 (recall-weighted)": fbeta_score(y, pred, beta=2, zero_division=0),
        "MCC": matthews_corrcoef(y, pred),
        "false alarm rate": fp / (fp + tn),
        "miss rate": fn / (fn + tp),
        # raw counts
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "borderline caught": f"{(border > thr).sum()}/{len(border)}",
    }

In [ ]:
results = {name: summarize(a, r) for name, (a, r) in all_scores.items()}
results_df = pd.DataFrame(results).T.sort_values("ROC AUC", ascending=False)
results_df.style.format(precision=3)

In [ ]:
a, r = all_scores["Robust covariance"]
thr = a.quantile(0.95)
missed, caught = r[r <= thr].index, r[r > thr].index

sc = StandardScaler().fit(X[acc])
Zf = pd.DataFrame(sc.transform(X), index=X.index, columns=feat_cols)  # z-scores vs. good batches

pd.DataFrame({"caught": Zf.loc[caught].mean(),
              "missed": Zf.loc[missed].mean()}).sort_values("missed", key=abs, ascending=False).head(15)

In [ ]:
# phase features: split each batch into start / middle / end
df["phase"] = pd.cut(df["t_norm"] if "t_norm" in df else
                     df["t"] / df.groupby("batch_id")["t"].transform("max"),
                     bins=[-0.01, 0.2, 0.8, 1.0], labels=["start", "mid", "end"])
phase_feats = df.groupby(["batch_id", "phase"], observed=True)[features].mean().unstack()
phase_feats.columns = [f"{f}_{p}" for f, p in phase_feats.columns]

# robust spread + spikes
q = df.groupby("batch_id")[features].quantile([0.05, 0.95]).unstack()
q.columns = [f"{f}_q{int(p*100)}" for f, p in q.columns]

X2 = X.join(phase_feats).join(q)

In [ ]:
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,
                             balanced_accuracy_score, precision_score, recall_score,
                             f1_score, fbeta_score, matthews_corrcoef, confusion_matrix)

def summarize(acc_s, rej_s, q=0.95, thr=None):
    scores = pd.concat([acc_s, rej_s])
    y = scores.index.isin(rej_s.index).astype(int)      # 1 = rejected (positive class)
    thr = acc_s.quantile(q) if thr is None else thr     # fixed false-alarm rate, or a given threshold
    pred = (scores > thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    border = rej_s[rej_s.index.isin(BORDER)]

    return {
        "ROC AUC": roc_auc_score(y, scores),
        "PR AUC": average_precision_score(y, scores),
        "accuracy": accuracy_score(y, pred),
        "balanced accuracy": balanced_accuracy_score(y, pred),
        "precision": precision_score(y, pred, zero_division=0),
        "recall (sensitivity)": recall_score(y, pred, zero_division=0),
        "specificity": tn / (tn + fp),
        "F1": f1_score(y, pred, zero_division=0),
        "F2 (recall-weighted)": fbeta_score(y, pred, beta=2, zero_division=0),
        "MCC": matthews_corrcoef(y, pred),
        "false alarm rate": fp / (fp + tn),
        "miss rate": fn / (fn + tp),
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "borderline caught": f"{(border > thr).sum()}/{len(border)}",
    }

In [ ]:
def rank_ensemble(names):
    ranks = pd.concat([pd.concat(all_scores[n]).rank(pct=True) for n in names], axis=1).mean(axis=1)
    a0, r0 = all_scores[names[0]]
    return ranks[a0.index], ranks[r0.index]

all_scores["Ensemble"] = rank_ensemble(["Robust covariance", "One-class SVM", "LSTM autoencoder"])
summarize(*all_scores["Ensemble"])

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut, cross_val_predict

y_all = (batch_feats["label"] == "rejected").astype(int)
clf = make_pipeline(StandardScaler(),
                    LogisticRegression(class_weight="balanced", C=0.1, max_iter=2000))
p = cross_val_predict(clf, X, y_all, cv=LeaveOneOut(), method="predict_proba")[:, 1]
print("ROC AUC", roc_auc_score(y_all, p), "PR AUC", average_precision_score(y_all, p))

In [ ]:
class PCAMonitor(BaseEstimator):
    """Multivariate statistical process control: Hotelling T² + Q (SPE) residual."""
    def __init__(self, n_components=10, stat="combined"):
        self.n_components = n_components
        self.stat = stat
    def _raw(self, X):
        X = np.asarray(X); T = self.pca_.transform(X)
        t2 = (T ** 2 / self.pca_.explained_variance_).sum(axis=1)
        q = ((X - self.pca_.inverse_transform(T)) ** 2).sum(axis=1)
        return t2, q
    def fit(self, X, y=None):
        self.pca_ = PCA(self.n_components).fit(np.asarray(X))
        t2, q = self._raw(X)
        self.t2_med_, self.q_med_ = np.median(t2), np.median(q)   # put both on the same scale
        return self
    def score_samples(self, X):
        t2, q = self._raw(X)
        t2, q = t2 / self.t2_med_, q / self.q_med_
        return -{"T2": t2, "Q": q, "combined": t2 + q}[self.stat]

In [ ]:
class TopKZ(BaseEstimator):
    """Mean of the k largest robust z-scores (median/MAD, from good batches)."""
    def __init__(self, k=3):
        self.k = k
    def fit(self, X, y=None):
        X = np.asarray(X)
        self.med_ = np.median(X, axis=0)
        self.mad_ = 1.4826 * np.median(np.abs(X - self.med_), axis=0) + 1e-9
        return self
    def score_samples(self, X):
        z = np.abs((np.asarray(X) - self.med_) / self.mad_)
        return -np.sort(z, axis=1)[:, -self.k:].mean(axis=1)

In [ ]:
new_models = {
    "PCA T²":       make_pipeline(StandardScaler(), PCAMonitor(10, "T2")),
    "PCA Q":        make_pipeline(StandardScaler(), PCAMonitor(10, "Q")),
    "PCA T² + Q":   make_pipeline(StandardScaler(), PCAMonitor(10, "combined")),
    "Top-3 z":      TopKZ(k=3),
}
for name, model in new_models.items():
    print("running", name, "...")
    all_scores[name] = evaluate(model, X, acc)

all_scores["Ensemble v2"] = rank_ensemble(["Robust covariance", "PCA Q", "Top-3 z"])

results = {n: summarize(a, r) for n, (a, r) in all_scores.items()}
pd.DataFrame(results).T.sort_values("PR AUC", ascending=False).style.format(precision=3)

In [ ]:
status = pd.Series("accepted", index=X.index)
status[caught], status[missed] = "rejected – caught", "rejected – missed"

sns.scatterplot(x=X["density_slope"], y=X["temperature_1_std"], hue=status,
                palette={"accepted": "grey", "rejected – caught": "red",
                         "rejected – missed": "orange"}, s=50)
plt.title("Two failure modes in process A"); plt.show()

In [ ]:
a, r = all_scores["PCA T²"]
thr = a.quantile(0.95)
missed_t2, caught_t2 = r[r <= thr].index, r[r > thr].index

print("newly caught vs. robust covariance:", sorted(set(caught_t2) - set(caught)))
print("still missed:", len(missed_t2))

pd.DataFrame({"caught": Zf.loc[caught_t2].median(),
              "missed": Zf.loc[missed_t2].median()}).sort_values("missed", key=abs, ascending=False).head(15)

In [ ]:
for k in [3, 5, 8, 10, 15, 20, 30]:
    a_k, r_k = evaluate(make_pipeline(StandardScaler(), PCAMonitor(k, "T2")), X, acc)
    s = summarize(a_k, r_k)
    print(f"k={k:>2}  ROC {s['ROC AUC']:.3f}  PR {s['PR AUC']:.3f}  caught {s['TP']}/40  MCC {s['MCC']:.3f}")

In [ ]:
p = PCA().fit(StandardScaler().fit_transform(X[acc]))
print("components for 90% variance:", np.searchsorted(np.cumsum(p.explained_variance_ratio_), 0.90) + 1)

In [ ]:
for name, Xf in [("base features", X), ("+ phase & quantiles", X2)]:
    a_f, r_f = evaluate(make_pipeline(StandardScaler(), PCAMonitor(10, "T2")), Xf, acc)
    s = summarize(a_f, r_f)
    print(f"{name:<22} ROC {s['ROC AUC']:.3f}  PR {s['PR AUC']:.3f}  caught {s['TP']}/40")

In [ ]:
def cost_threshold(acc_s, rej_s, cost_fn=5, cost_fp=1):
    """Threshold that minimises cost_fn × misses + cost_fp × false alarms."""
    scores = pd.concat([acc_s, rej_s])
    y = scores.index.isin(rej_s.index)
    cands = np.unique(scores)
    costs = [cost_fn * ((scores <= t) & y).sum() + cost_fp * ((scores > t) & ~y).sum()
             for t in cands]
    return cands[int(np.argmin(costs))]

In [ ]:
a, r = all_scores["PCA T²"]
for ratio in [1, 3, 5, 10]:
    s = summarize(a, r, thr=cost_threshold(a, r, cost_fn=ratio))
    print(f"miss = {ratio}× false alarm → caught {s['TP']}/40, false alarms {s['FP']}/121")

In [ ]:
status = pd.Series("accepted", index=X.index)
status[caught_t2], status[missed_t2] = "rejected – caught", "rejected – missed"

sns.scatterplot(x=X["temperature_1_mean"], y=X["temperature_1_std"], hue=status,
                palette={"accepted": "grey", "rejected – caught": "red",
                         "rejected – missed": "orange"}, s=50)
plt.axhline(X.loc[acc, "temperature_1_std"].max(), ls="--", color="k", lw=0.8,
            label="max of accepted batches")
plt.legend(); plt.title("Process A: PCA T² detections"); plt.show()